In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from uncertainties import ufloat as uf
from uncertainties import unumpy as unp
from pathlib import Path

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [19]:
# definice funkci

# nejistota typu A 
def unc_A(data):
    '''
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

# nacteni z df do numpy
def df_to_numpy(df, sloupce, od=0, do=None, dtype=float):
    """
    sloupce: jmeno sloupce (str) nebo seznam jmen (list[str])
    od, do:  rozsah radku podle pozice (od vcetne, do bez); do=None znamená až do konce
    """
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    data = df[sloupce].iloc[od:do].to_numpy(dtype=dtype)
    return data[:, 0] if len(sloupce) == 1 else data

# nacteni z df do jednotlivych arrays
def df_to_arrays(df, sloupce, od=0, do=None, dtype=float):
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    return [df[s].iloc[od:do].dropna().to_numpy(dtype=dtype) for s in sloupce]

# ufloat pro tuple
def tuple_unc_A(*velicina):
    list_velicin = [*velicina]
    return tuple(uf(np.mean(v), unc_A(v)) for v in list_velicin)

# array do latex
def array_to_latex(*array, dig=4, format='f', style='L'):
    for u in array:
        print(f'${u:.{dig}{format}{style}}$')

In [50]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "0"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_tenk = pd.read_csv(url)
df_tenk.dropna()

,Číslo proužku,Místo na vzorku,Počet interferenčních proužků,x1a [mm],x2a [mm],Místo na vzorku.1,Počet interferenčních proužků.1,x1b [mm],x2b [mm],Místo na vzorku.2,...,Počet interferenčních proužků.5,x1f [mm],x2f [mm],Počet interferenčních proužků.6,x1g [mm],x2g [mm],Místo na vzorku.6,Počet interferenčních proužků.7,x1h [mm],x2h [mm]


In [48]:
# tenka vrstva
vln_delka_tenk = 532.1*1e-9
df_tenk_list = ['x1a [mm]', 'x2a [mm]', 'x1b [mm]', 'x2b [mm]', 'x1c [mm]', 'x2c [mm]', 'x1d [mm]', 'x2d [mm]', 'x1e [mm]', 'x2e [mm]', 'x1f [mm]', 'x2f [mm]', 'x1g [mm]', 'x2g [mm]', 'x1h [mm]', 'x2h [mm]']
x1a, x2a, x1b, x2b, x1c, x2c, x1d, x2d, x1e, x2e, x1f, x2f, x1g, x2g, x1h, x2h = df_to_arrays(df_tenk, df_tenk_list, dtype=int)

def tenka(pripona):
    x1 = globals()[f"x1{pripona}"]
    x2 = globals()[f"x2{pripona}"]
    return (x2 / x1) * (vln_delka_tenk / 2)

from string import ascii_lowercase

n=8
vysledek = []
for k in ascii_lowercase[:n]:
    vysledek.extend(tenka(k))

tlouska = uf(np.mean(vysledek), unc_A(vysledek))
array_to_latex(tlouska, dig=1, format='u')

$\left(1.00 \pm 0.03\right) \times 10^{-7}$


In [ ]:
# nacteni google tabulek
sheet_id = "1kPY11z8IlfFcIX7OrcH3VEfNgLK5n-SDjEpLverEX_E"
gid = "1791918306"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_mrizka = pd.read_csv(url)
df_mrizka

In [ ]:
# difrakce na mrizce
vln_delka_dif = 632.8*1e-9

df_mrizka_list = ['Vzdálenost mřížka–stínítko xa [mm]', 'y1a vpravo [mm]', 'y1a vlevo [mm]', 'y2a vpravo [mm]', 'y2a vlevo [mm]', 'Vzdálenost mřížka–stínítko xb [mm]', 'y1b vpravo [mm]', 'y1b vlevo [mm]', 'y2b vpravo [mm]', 'y2b vlevo [mm]']
xa, y1a_right, y1a_left, y2a_right, y2a_left, xb, y1b_right, y1b_left, y2b_right, y2b_left = df_to_numpy(df_mrizka, df_mrizka_list, dtype=int).T
y1a = (y1a_left + y1a_right)/2
y2a = (y2a_left + y2a_right)/2
y1b = (y1b_left + y1b_right)/2
y2b = (y2b_left + y2b_right)/2

def difraction_formule(y, x, m):
    sin_alpha = y/((y**2 + x**2)**0.5)
    d = (m * vln_delka_dif)/sin_alpha
    return d

d_a = np.concatenate([difraction_formule(y1a, xa, 1), difraction_formule(y2a, xa, 2)])
d_a = uf(np.mean(d_a), unc_A(d_a))
array_to_latex(d_a, dig=1, format='u')
array_to_latex(1/d_a/1000, dig=1)
d_b = np.concatenate([difraction_formule(y1b, xb, 1), difraction_formule(y2b, xb, 2)])
d_b = uf(np.mean(d_b), unc_A(d_b))
array_to_latex(d_b, dig=1, format='u')
array_to_latex(1/d_b/1000, dig=1)